In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spatialdata as sd
import squidpy as sq
import scanpy as sc
import warnings
import joblib
import decoupler as dc
from pathlib import Path
# from adjustText import adjust_text
from scipy.stats import spearmanr

import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from joblib import Parallel, delayed

from gbmcollaboration.viz.figure_helpers import set_mpl_rcparams,savefig
from gbmcollaboration.viz import dotplot, volcano
# from gbmcollaboration.utils.coda_utils import compute_logit_props, compare_proportions
from gbmcollaboration.utils import (
    xenium_io_utils as xio,
    xenium_utils as xu
)

set_mpl_rcparams()

def pseudobulk_lognorm(adata, by):
    pb = sc.get.aggregate(adata, by=by, func='sum',layer='counts' )
    pb.X = pb.layers['sum']
    sc.pp.normalize_total(pb)
    sc.pp.log1p(pb)
    return pb

def run_ulm_smooth(adata,net):
    ''' Smooth .X values with neighbors before computing ULM'''
    adata.layers['X'] = adata.X.copy()
    adata.X = xu.agg_neighbor_values(adata.obsp['spatial_connectivities'],adata.to_df()).values
    dc.run_ulm(adata,net=net,min_n=1,use_raw=False,weight=None)
    adata.X = adata.layers.pop('X')

def spearman_cols(df, y_cols, x_col):
    out = []
    for c in y_cols:
        r, p = spearmanr(df[x_col], df[c], nan_policy='omit')
        out.append((c, r, p))
    return pd.DataFrame(out, columns=['category', 'rho', 'pval'])

def assign_grid_tiles(adata, sample_key='sample', spatial_key='spatial', tile_size=55.0):
    """
    Assigns each cell to a non-overlapping spatial grid tile.
    Adds 'tile_id' to adata.obs.
    """
    x = adata.obsm[spatial_key][:, 0]
    y = adata.obsm[spatial_key][:, 1]
    
    # Calculate discrete grid indices
    tile_x = np.floor(x / tile_size).astype(int).astype(str)
    tile_y = np.floor(y / tile_size).astype(int).astype(str)
    
    # Create unique tile IDs. We prepend the sample ID 
    samples = adata.obs[sample_key].astype(str)
    tile_ids = samples + "_tile_" + tile_x + "_" + tile_y
    
    adata.obs['tile_id'] = pd.Categorical(tile_ids)

def plot_correlation_heatmap(
    corr_df,
    index_col='sample',
    columns_col='category',
    values_col='rho',
    pval_col='pval',
    figsize=(12, 8),
    cmap="vlag",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_label="Spearman rho",
    title="Per-sample Spearman correlation",
    title_fontsize=14,
    xtick_rotation=45,
    ytick_rotation=0,
    sort_by_mean=True,
    significance_threshold=0.05,
    mask_non_significant=True,
):
    """
    Plot a heatmap of per-sample Spearman correlations, with optional sorting and significance masking.

    Parameters:
    - corr_df: DataFrame containing correlation data (must include 'sample', 'category', 'rho', and 'pval' columns).
    - index_col: Column name for the index (e.g., 'sample').
    - columns_col: Column name for the columns (e.g., 'category').
    - values_col: Column name for the correlation values (e.g., 'rho').
    - pval_col: Column name for the p-values (e.g., 'pval').
    - figsize: Figure size.
    - cmap: Colormap for the heatmap.
    - center: Center value for the colormap.
    - annot: Whether to annotate the heatmap cells.
    - fmt: Format string for annotations.
    - linewidths: Width of the lines between cells.
    - cbar_label: Label for the colorbar.
    - title: Title for the plot.
    - title_fontsize: Font size for the title.
    - xtick_rotation: Rotation for x-axis ticks.
    - ytick_rotation: Rotation for y-axis ticks.
    - sort_by_mean: Whether to sort columns by mean correlation.
    - significance_threshold: Threshold for significance (default: 0.05).
    - mask_non_significant: Whether to mask non-significant values.
    """

    # Pivot to wide format
    corr_wide = corr_df.pivot(index=index_col, columns=columns_col, values=values_col)

    # Sort columns by mean correlation
    if sort_by_mean:
        corr_wide = corr_wide[corr_wide.mean().sort_values(ascending=False).index]

    # Create significance mask
    sig_mask = corr_df.pivot(index=index_col, columns=columns_col, values=pval_col) >= significance_threshold

    # Plot
    plt.figure(figsize=figsize)
    ax = sns.heatmap(
        corr_wide,
        cmap=cmap,
        center=center,
        annot=annot,
        fmt=fmt,
        linewidths=linewidths,
        cbar_kws={'label': cbar_label},
        mask=sig_mask if mask_non_significant else None,
    )

    plt.xticks(rotation=xtick_rotation, ha='right')
    plt.yticks(rotation=ytick_rotation)
    plt.title(title, fontsize=title_fontsize)
    plt.tight_layout()

    return ax


def create_non_overlapping_grid(df_cells, tile_size=55.0, min_cells_per_tile=1):
    """
    Converts single-cell spatial data into a non-overlapping regular grid.
    
    Parameters:
    - df_cells: DataFrame containing 'x', 'y', 'is_TIL', 'ECM_score', etc.
    - tile_size: Size of the square tile in the same units as x/y (e.g., 55 microns).
    - min_cells_per_tile: Minimum number of cells required to keep a tile for analysis.
    
    Returns:
    - df_tiles: DataFrame where each row is a tile with aggregated metrics.
    """
    df = df_cells.copy()
    
    # 1. Assign each cell to a discrete grid index based on x/y coordinates
    df['tile_x_idx'] = np.floor(df['x'] / tile_size).astype(int)
    df['tile_y_idx'] = np.floor(df['y'] / tile_size).astype(int)
    
    # Create a unique string ID for each tile
    df['tile_id'] = df['tile_x_idx'].astype(str) + "_" + df['tile_y_idx'].astype(str)
    
    # Filter out sparse or empty tiles
    df_tiles = df_tiles[df_tiles['total_cells'] >= min_cells_per_tile].copy()
    
    return df


def plot_grid(adata,til_key='is_TIL'):
    coords = adata.obsm['spatial']
    is_til = adata.obs[til_key]

    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 10))

    # 1. Scatter the non-TIL cells (gray)
    ax.scatter(
        coords[is_til == 0, 0], 
        coords[is_til == 0, 1], 
        c='lightgray', s=1, alpha=0.5, label='Other Cells'
    )

    # 2. Scatter the TIL cells (red)
    ax.scatter(
        coords[is_til == 1, 0], 
        coords[is_til == 1, 1], 
        c='red', s=3, alpha=0.9, label='TILs'
    )

    # 3. Draw the grid lines
    tile_size = 55.0
    x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
    y_min, y_max = coords[:, 1].min(), coords[:, 1].max()

    # Calculate the precise grid line locations based on how we did np.floor() earlier
    x_lines = np.arange(np.floor(x_min / tile_size) * tile_size, x_max, tile_size)
    y_lines = np.arange(np.floor(y_min / tile_size) * tile_size, y_max, tile_size)

    # Add vertical and horizontal lines
    for xl in x_lines:
        ax.axvline(xl, color='black', linewidth=0.5, alpha=0.3)
    for yl in y_lines:
        ax.axhline(yl, color='black', linewidth=0.5, alpha=0.3)

    ax.set_aspect('equal')
    ax.set_title(f"Spatial Grid Overlay (Tile Size: {tile_size}µm)\nSample: {sample_id}")
    ax.legend(markerscale=5)
    plt.show()


def spearman_tils(pb, categories, sample_key, til_count_key='TIL_count', 
                  min_tiles=10, filter_zero_til=True):
    """
    Computes Spearman correlation per sample and applies FDR correction per sample.
    """
    # 1. Prepare data
    pb_df = pb.obs.copy() # Work directly on obs DataFrame
    if filter_zero_til:
        pb_df = pb_df[pb_df[til_count_key] > 0]

    # 2. Correlate per sample
    results = []
    for sample_id, group in pb_df.groupby(sample_key, observed=True):
        if len(group) < min_tiles:
            continue
        
        for cat in categories:
            rho, p_val = spearmanr(group[til_count_key], group[cat], nan_policy='omit')
            results.append({
                sample_key: sample_id,
                'category': cat,
                'n_tiles': len(group),
                'rho': rho,
                'p': p_val
            })

    res_df = pd.DataFrame(results)


    # 3. Correct p-values per sample
    def fdr_correct(group):
        # Drop NaNs before correction
        valid = group['p'].notna()
        p_adj = np.full(len(group), np.nan)
        if valid.any():
            _, corrected, _, _ = multipletests(group.loc[valid, 'p'], method='fdr_bh')
            p_adj[valid] = corrected
        group['p_adj'] = p_adj
        return group
    
    res_df = res_df.groupby(sample_key, group_keys=False).apply(fdr_correct)

    return res_df

## Parameters


In [ ]:
# path to ecm metadata
path_ecm_csv = '/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/supplementary_tables/4a_ECM_genes.csv'

# path to raw and processed xenium
results_dir = Path('/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/')
figures_dir = Path('/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/figures/')
base_dir = Path('/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/')

# folder structure
xenium_levels = ['segmentation','condition','panel','donor','sample']

# data parameters
correction_methods = ['raw','split_fully_purified'] # correction of count matrices after segmentation
segmentation = '10x_mm_5um' # or '10x_mm_5um' or 'proseg_expected' or 'all'
condition = 'gbm'
panel = 'hBrain_v1.2' # 'hAtlas_v1.1' or 'hBrain_v1.2' or 'all'

# cell type annotation parameters
normalisation = 'lognorm'
reference = 'allcenters_10k_ct'  # annotated with all centers snRNA ref, subsampled to 10k max per cell type
method = 'rctd_class_aware'
level = None # celltype_curated_level1,2,3,4 available
level_keys = [f'celltype_level{i}_curated_first_type' for i in range(1,5)]

# SPLIT parameters
split_level='celltype_level2_curated'

# fixed params
BATCH_KEY = 'dataset_id'
SPATIAL_KEY = 'spatial'

read_kwargs = dict(
    # set anndata = False to load in spatialdata format instead of a single anndata
    # (allows to load images, transcripts, etc)
    anndata = True,

    # extra parameters if anndata = False
    cells_as_circles = False,
    cells_boundaries = False,
    cells_boundaries_layers = False, # for proseg only
    nucleus_boundaries = False,
    cells_labels = False,
    nucleus_labels = False,
    transcripts = False,
    morphology_mip = False,
    morphology_focus = False,
    aligned_images = False,
    gex_only = True,
    pool_mode="thread",
    max_workers=6
)

# path to save figures
# sc.settings.figdir = Path(f'/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/figures/xenium/{correction_method}/{segmentation}')
# sc.settings.figdir.mkdir(parents=True,exist_ok=True)
# sc.set_figure_params(figsize=(8,8))

## Read and preprocess data

In [ ]:
p = results_dir/f'xenium/ad_rna_{panel}_{segmentation}.h5ad'

if False:#p.exists(): 
    adata_dict = joblib.load(p)

else:
    xenium_paths, xenium_annot_paths = xio.discover_xenium_paths(
        base_dir=base_dir,
        normalisation=normalisation,
        reference=reference,
        method=method,
        level=None,
        load_correction=True,
        split_level='celltype_level2_curated',
        segmentations_filter=[segmentation],
        conditions_filter=[condition] if condition != 'all' else None,
        panels_filter=[panel] if panel != 'all' else None,
    )

    # Read samples
    xds = xio.read_count_correction_samples(xenium_paths,['split_fully_purified'])
    xds['raw'] = xio.read_xenium_samples(xenium_paths['raw'],**read_kwargs)

    # add cell type annotation from raw to all correction methods
    xio.read_annotations(xds, correction_methods, xenium_annot_paths, level, max_workers=read_kwargs['max_workers'])

    # add per sample umaps
    for corr, xds_corr in xds.items():
        for k, xd in xds_corr.items():
            print(corr,k)
            xd.layers['counts'] = xd.X.copy()
            # sc.pp.normalize_total(xd)
            # sc.pp.log1p(xd)
            # sc.pp.pca(xd,key_added='X_rna_pca_sample')
            # sc.pp.neighbors(xd,use_rep='X_rna_pca_sample')
            # sc.tl.umap(xd,key_added='X_rna_umap_sample')

            if corr != 'raw':
                xd.obsm['spatial'] = xds['raw'][k][xd.obs_names].obsm['spatial']

    # concat samples into a single anndata
    adata_dict = {}
    if read_kwargs['anndata']:
        if not read_kwargs['gex_only']: 
            
            modalities = {'Gene Expression':'rna','Protein Expression':'prot'}

            for corr in correction_methods:
                if corr == 'raw':
                    adata_dict[corr] = xio.concat_by_modality(xds[corr],label='dataset_id', join='outer', modalities=modalities)
                else:
                    adata_dict[corr] = {}
                    adata_dict[corr]['rna'] = sc.concat({k: v for k, v in xds[corr].items() },label='dataset_id', join='outer')

                for k, ad_ in adata_dict[corr].items():
                    ad_.obs[xenium_levels] = pd.DataFrame(ad_.obs['dataset_id'].tolist(),index=ad_.obs.index,columns=xenium_levels)
                    ad_.obs['correction_method'] = corr
        else:
            for corr in correction_methods:
                ad_ = sc.concat({k: v for k, v in xds[corr].items() },label='dataset_id', join='outer')
                ad_.obs[xenium_levels] = pd.DataFrame(ad_.obs['dataset_id'].tolist(),index=ad_.obs.index,columns=xenium_levels)
                ad_.obs['correction_method'] = corr
                adata_dict[corr] = {'rna':ad_}

    else:
        # dont concat if sdatas (anndata=False)
        pass

    # info about the loaded cohort
    df_cohort_info = pd.DataFrame(
        [k+(corr_method,)
        for corr_method in correction_methods
        for k in xds[corr_method].keys() ],
        columns=xenium_levels+['correction_method']
    )
    print(f'Loaded {len(df_cohort_info)} samples:')
    display(df_cohort_info)

    # save loaded cohort
    p.parent.mkdir(exist_ok=True,parents=True)
    joblib.dump(adata_dict,p)

### prepare data for analysis

In [ ]:
# analysis parameters
corr = 'split_fully_purified'
til_cell_types = ['T_cell', 'NK']
sample_key = 'sample'
cluster_key = level_keys[2]
radius = 55/2

# get adatas per modality and remove cells with NaN annotations
ad_rna =  adata_dict[corr]['rna']
has_annotation_idx = ad_rna.obs[level_keys].notna().all(1)

ad_rna = ad_rna[has_annotation_idx].copy()
if 'prot' in adata_dict['raw']:
    ad_prot =  adata_dict['raw']['prot'][has_annotation_idx]

# lognormalize
if 'log1p' not in ad_rna.uns:
    sc.pp.normalize_total(ad_rna)
    sc.pp.log1p(ad_rna)

# get spatial rNN, prep .obs
for k in [sample_key,cluster_key]:
    ad_rna.obs[k] = ad_rna.obs[k].astype('category')

ad_rna.obs['TIL_status_bool'] = ad_rna.obs[cluster_key].isin(til_cell_types)
ad_rna.obs['TIL_status'] = np.where(ad_rna.obs['TIL_status_bool'],'TIL','non-TIL')

sq.gr.spatial_neighbors(ad_rna, radius=radius, library_key=sample_key, coord_type='generic', set_diag=True)

# add some stats in .obs and find spatial clusters based on TILs
xu.find_spatial_clusters(ad_rna, 
    score_key=cluster_key, 
    score_threshold=1, 
    min_cluster_size=10, 
    radius=radius,
    prefix='TIL', 
    score_cats=til_cell_types)

ad_rna.obs['TIL_sum_cat'] = pd.cut(
    ad_rna.obs['TIL_sum'],
    bins=[-1, 1, 2, np.inf],
    labels=['non-TIL', 'low-TIL', 'high-TIL']
)

# assign grid tiles
assign_grid_tiles(ad_rna, sample_key=sample_key, spatial_key='spatial', tile_size=55.0)

# read ecm metadata
df_ecm = pd.read_csv(path_ecm_csv)

# TIL analysis

## TIL pseudobulk vs rest

In [ ]:
groupby = 'TIL_status'
pb_rna = pseudobulk_lognorm(ad_rna,by=[sample_key,groupby])
sc.tl.rank_genes_groups(pb_rna,groupby=groupby)
df_xe_rna_de = sc.get.rank_genes_groups_df(pb_rna,'TIL')

df_plot = (
    df_xe_rna_de
    .join(df_vis_de.set_index('gene'),on='names',rsuffix='_vis')
    .dropna(subset='logFC')
)
df_plot = df_plot.rename(columns={'logfoldchanges':'logFC_xenium','logFC':'logFC_visium'})
df_plot.to_csv(results_dir/f'xenium/de_results_TIL+_{panel}_{segmentation}.csv')

plot_cross_platform_comparison(df_plot,title='TIL+ vs TIL- logFC for Xenium and Visium')

## TIL spatial regions pseudobulk vs rest

### test high-TIL vs low-TIL spatial neighborhoods

In [ ]:
groupby = 'TIL_sum_cat'
ad_rna_til_pos = ad_rna[ad_rna.obs[groupby]!='non-TIL']
pb_rna = pseudobulk_lognorm(ad_rna_til_pos,by=[sample_key,groupby])
sc.tl.rank_genes_groups(pb_rna,groupby=groupby)
df_xe_rna_de = sc.get.rank_genes_groups_df(pb_rna,'high-TIL')

df_plot = (
    df_xe_rna_de
    .join(df_vis_de.set_index('gene'),on='names',rsuffix='_vis')
    .dropna(subset='logFC')
)
df_plot = df_plot.rename(columns={'logfoldchanges':'logFC_xenium','logFC':'logFC_visium'})
df_plot.to_csv(results_dir/f'xenium/de_results_high-TIL_vs_low-TIL_{panel}_{segmentation}.csv')

plot_cross_platform_comparison(df_plot,title='high-TIL vs low-TIL neighborhood logFC for Xenium and Visium')

### test high-TIL vs low-TIL spatial components

In [ ]:
# Compute TIL sums per component
til_sum_per_comp = (
    ad_rna.obs
    .groupby('TIL_split')[level_keys[2]]
    .value_counts()
    .unstack(fill_value=0)[til_cell_types]
    .sum(1)
)

# Map TIL_split to status based on summed counts
til_status_map = pd.Series(index=til_sum_per_comp.index, dtype=str)
til_status_map[til_sum_per_comp < 2] = 'non-TIL'
til_status_map[til_sum_per_comp == 2] = 'low-TIL'
til_status_map[til_sum_per_comp > 2] = 'high-TIL'

# Assign to obs
ad_rna.obs['TIL_status_region'] = ad_rna.obs['TIL_split'].map(til_status_map)

groupby = 'TIL_status_region'
ad_rna_til_pos = ad_rna[ad_rna.obs[groupby]!='non-TIL']
pb_rna = pseudobulk_lognorm(ad_rna_til_pos,by=[sample_key,groupby])
sc.tl.rank_genes_groups(pb_rna,groupby=groupby)
df_xe_rna_de = sc.get.rank_genes_groups_df(pb_rna,'high-TIL')

df_plot = (
    df_xe_rna_de
    .join(df_vis_de.set_index('gene'),on='names',rsuffix='_vis')
    .dropna(subset='logFC')
)
df_plot = df_plot.rename(columns={'logfoldchanges':'logFC_xenium','logFC':'logFC_visium'})
df_plot.to_csv(results_dir/f'xenium/de_results_high-TIL_vs_low-TIL_{panel}_{segmentation}.csv')

plot_cross_platform_comparison(df_plot,title='high-TIL vs low-TIL regions logFC for Xenium and Visium')

In [ ]:
dfs_comp = compute_logit_props(ad_rna.obs, level_keys[2], groupby=[sample_key,'TIL_status_region'])
df_props =  dfs_comp['props'].drop(index='non-TIL', level='TIL_status_region')
grouping = df_props.index.to_frame().set_index(sample_key).squeeze().astype(str)
df_props.index = df_props.index.get_level_values(sample_key)
df_props_de = compare_proportions(
    df_props,
    grouping=grouping,
    test_group='high-TIL',
    ref_group='low-TIL'
)

f, axs = plt.subplots(1,2,figsize=(15,7))
volcano(df_props_de,logfc_col='log2fc',pval_col="padj", fc_thresh=1, ax=axs[0])
volcano(df_props_de,logfc_col='effect_size',pval_col="padj",fc_thresh=0.2, ax=axs[1])
plt.show()

In [ ]:
df_plot = df_props.copy()
df_plot.index = grouping.values
ax = df_plot.plot(kind='bar', stacked=True)
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    borderaxespad=0
)
plt.tight_layout()

# Correlations with TIL sum

## Spearman pseudobulk

### cell type composition

In [ ]:
A = ad_rna.obsp['spatial_connectivities']
cluster_key_ = level_keys[-1]
df_ct_region_counts = xu.agg_neighbor_values(A, ad_rna.obs[cluster_key_])
df_ct_region_counts['TIL_sum'] = ad_rna.obs['TIL_sum'].values
df_ct_region_counts[sample_key] = ad_rna.obs[sample_key].values
df_ct_region_counts_til_pos = df_ct_region_counts[df_ct_region_counts['TIL_sum']>0].copy()

cats = ad_rna.obs[cluster_key_].unique().tolist()

corr_df = (
    df_ct_region_counts_til_pos
    .groupby('sample')
    .apply(spearman_cols, cats,x_col='TIL_sum')
    .reset_index(level=1, drop=True)
    .reset_index()
)

# pivot to wide format
corr_wide = corr_df.pivot(
    index='sample',
    columns='category',
    values='rho'
)

# optional: sort columns by overall mean correlation
corr_wide = corr_wide[corr_wide.mean().sort_values(ascending=False).index]

sig_mask = corr_df.pivot(index='sample', columns='category', values='pval') < 0.05

plt.figure(figsize=(30, 8))
ax = sns.heatmap(
    corr_wide,
    cmap="vlag",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={'label': "Spearman rho"}
)

# overlay black dots for significant correlations
# for i, sample in enumerate(corr_wide.index):
#     for j, ecm in enumerate(corr_wide.columns):
#         if sig_mask.loc[sample, ecm]:
#             ax.plot(j + 0.5, i + 0.5, 'o', color='black', markersize=5)
plt.yticks([])
plt.xticks(rotation=45, ha='right')
plt.title("Per-sample Spearman correlation", fontsize=14)
plt.tight_layout()
plt.show()

### ECM 

In [ ]:
dc.run_ulm(ad_rna,net=df_ecm,min_n=1,use_raw=False,weight=None)

df_ulm = ad_rna.obsm['ulm_estimate']
ad_rna.obs = ad_rna.obs.assign(**df_ulm)
ecm_cats = df_ulm.columns

In [ ]:
for k in level_keys:
    df_ulm_mean = ad_rna.obs.groupby([k,'sample'])[ecm_cats].mean().reset_index()
    df_long = df_ulm_mean.melt(
        id_vars=[k, 'sample'],value_vars=ecm_cats,
        var_name='ecm_category', value_name='mean_expression'
    )
    order = (
        df_long.groupby(k)['mean_expression']
        .mean().sort_values(ascending=False).index
    )
    df_plot = df_long

    plt.figure(figsize=(8, df_plot[k].nunique()//1.5))

    ax = sns.boxplot(
        data=df_long,
        y=k,x='mean_expression',hue='ecm_category',
        order=order,
        showfliers=False
    )

    sns.stripplot(
        data=df_long,
        y=k,x='mean_expression',hue='ecm_category',
        order=order,
        dodge=True,
        color='black',
        size=2.5,
        alpha=0.6,
        ax=ax
    )

    handles, labels = ax.get_legend_handles_labels()
    n = len(ecm_cats)

    ax.legend(
        handles[:n],
        labels[:n],
        title="ECM category",
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        frameon=False
    )

    ax.set_xlabel("")
    ax.set_ylabel("Mean expression per sample")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

#### spearman correlation with TIL sum

In [ ]:
# get smoothed ECM score
run_ulm_smooth(ad_rna,net=df_ecm)
df_ulm_til_pos = ad_rna.obsm['ulm_estimate']
ad_rna.obs = ad_rna.obs.assign(**df_ulm_til_pos)
ad_rna_til_pos = ad_rna[ad_rna.obs['TIL_sum']>0].copy()
ecm_cats = df_ulm_til_pos.columns.tolist()

In [ ]:
# get spearman correlation between TIL sum and ECM score
corr_df = (
    ad_rna_til_pos.obs
    .groupby('sample')
    .apply(spearman_cols, x_col='TIL_sum',y_cols=ecm_cats)
    .reset_index(level=1, drop=True)
    .reset_index()
)

plot_correlation_heatmap(corr_df)

## Spearman pseudobulk tiles

In [ ]:
cell_type_key = level_keys[-1]
til_key = 'TIL_status_bool'
min_n_cells_per_tile = 5
min_n_tiles_per_sample = 10

# Calculate tile-level metadata from the single-cell adata
tile_meta = ad_rna.obs.groupby('tile_id', observed=True).agg(
    total_cells=('tile_id', 'size'),   
    TIL_count=(til_key, 'sum')
)
cluster_counts = (
    ad_rna.obs.groupby('tile_id', observed=True)[cell_type_key]
    .value_counts()
    .unstack(fill_value=0)
)
tile_meta = tile_meta.join(cluster_counts)

# pseudobulk by tile
pb = pseudobulk_lognorm(ad_rna, by=[sample_key,'tile_id'])

# Filter out tiles with fewer than n cells
valid_tiles = tile_meta[tile_meta['total_cells'] >= min_n_cells_per_tile].index
tile_meta = tile_meta.loc[valid_tiles]
pb = pb[pb.obs['tile_id'].isin(valid_tiles)].copy()

# Add the cell counts and TIL counts back into the pseudobulked AnnData.obs
pb.obs = pb.obs.join(tile_meta,on='tile_id')

# Calculate ECM ULM scores 
dc.run_ulm(pb,net=df_ecm,min_n=1,use_raw=False,weight=None)
df_ulm = pb.obsm['ulm_estimate']
pb.obs = pb.obs.assign(**df_ulm)

# Categories to test
cell_type_cats = ad_rna.obs[cell_type_key].unique().tolist()
ecm_cats = df_ulm.columns

In [ ]:
# # plot example
# sample_id = ad_rna.obs[sample_key].unique()[0]
# mask = ad_rna.obs[sample_key] == sample_id
# ad_sample = ad_rna[mask]

# plot_grid(ad_sample)

### cell type composition

In [ ]:
# correlate filtering out TIL count = 0
res_df = spearman_tils(pb, cell_type_cats, sample_key, til_count_key='TIL_count', min_tiles=min_n_tiles_per_sample, filter_zero_til=True)
res_df = res_df.join(ad_rna.obs.set_index(sample_key)['donor'].drop_duplicates(),on=sample_key)

# plot
save_path=Path(figures_dir/f'xenium/{panel}/{segmentation}/{cell_type_key}_spearman_tiles.png')
save_path.parent.mkdir(exist_ok=True,parents=True)

x_order = res_df.groupby('category')['rho'].mean().sort_values(ascending=False).index

dotplot(res_df,
    x_col='category',
    y_col='donor',
    diff_col='rho',
    pval_col='p_adj',
    x_order=x_order,
    y_order=np.unique(res_df['donor']),
    figsize=(18, 5),
    # yticklabels=False,
    cbar_title='Spearman ρ',
    bbox_to_anchor_dot = (1.08, .8),
    min_dot_size=50,
    max_dot_size=200,
    center=0,
    save_path=None
)

savefig(res_df,save_path)

### ECM

In [ ]:
# correlate filtering out TIL count = 0
res_df = spearman_tils(pb, ecm_cats, sample_key, til_count_key='TIL_count', min_tiles=min_n_tiles_per_sample, filter_zero_til=True)
res_df = res_df.join(ad_rna.obs.set_index(sample_key)['donor'].drop_duplicates(),on=sample_key)

# plot
save_path=Path(figures_dir/f'xenium/{panel}/{segmentation}/ecm_spearman_tiles.png')
save_path.parent.mkdir(exist_ok=True,parents=True)

x_order = res_df.groupby('category')['rho'].mean().sort_values(ascending=False).index

dotplot(res_df,
    x_col='category',
    y_col='donor',
    diff_col='rho',
    pval_col='p_adj',
    x_order=x_order,
    y_order=np.unique(res_df['donor']),
    figsize=(10, 5),
    # yticklabels=False,
    cbar_title='Spearman ρ',
    bbox_to_anchor_dot = (1.15, .8),
    # min_dot_size=50,
    # max_dot_size=400,
    save_path=None
)
savefig(res_df,save_path)

## Spatial regression

### cell type composition

In [ ]:
# analysis
cluster_key_ = level_keys[-1]

A = ad_rna.obsp['spatial_connectivities']
df_ct_region_counts = xu.agg_neighbor_values(A, ad_rna.obs[cluster_key_])
df_ct_region_counts['TIL_sum'] = ad_rna.obs['TIL_sum'].values
df_ct_region_counts[sample_key] = ad_rna.obs[sample_key].values
df_ct_region_counts_til_pos = df_ct_region_counts[df_ct_region_counts['TIL_sum']>0].copy()

df_ct_region_counts_til_pos[['x','y']] = ad_rna.obsm['spatial']
df_ct_region_counts_til_pos['is_TIL'] = ad_rna.obs[cluster_key].isin(til_cell_types)
cats = ad_rna.obs[cluster_key_].unique().tolist()

analyzer = xu.SpatialTissueAnalyzer(radius=radius, n_perms=100) 

res_df = {}
summary_df = {}
for target_col in cats:
    print(target_col)

    res_df[target_col], summary_df[target_col] = analyzer.run_meta(
        df_ct_region_counts_til_pos, 
        sample_col='sample',
        source_col='is_TIL', 
        target_col=target_col, 
        method='spearman_spatial_perm',
        smooth_target=False,
    )

res_df = (
    pd.concat(res_df)
    .reset_index(level=1, drop=True)
    .reset_index().rename(columns={'index':'category'})
)

res_df = res_df.join(ad_rna.obs.set_index('sample')['donor'].drop_duplicates(),on='sample')

In [ ]:
save_path=Path(figures_dir/f'xenium/{panel}/{segmentation}/{cluster_key_}_spearman.png')
save_path.parent.mkdir(exist_ok=True,parents=True)

x_order = res_df.groupby('category')['rho'].mean().sort_values(ascending=False).index

dotplot(res_df,
    x_col='category',
    y_col='donor',
    diff_col='rho',
    x_order=x_order,
    y_order=np.unique(res_df['donor']),
    figsize=(18, 5),
    # yticklabels=False,
    bbox_to_anchor_dot = (1.08, .8),
    min_dot_size=20,
    max_dot_size=100,
    save_path=save_path
    # max_dot_size=40
)

### ECM 

In [ ]:
# get smoothed ECM score
run_ulm_smooth(ad_rna,net=df_ecm)
df_ulm_til_pos = ad_rna.obsm['ulm_estimate']
ad_rna.obs = ad_rna.obs.assign(**df_ulm_til_pos)
ad_rna_til_pos = ad_rna[ad_rna.obs['TIL_sum']>0].copy()
ecm_cats = df_ulm_til_pos.columns.tolist()

In [ ]:
# analysis
ad = ad_rna_til_pos
df_analysis = ad.obs.copy()
df_analysis[['x','y']] = ad.obsm['spatial']
df_analysis['is_TIL'] = ad.obs[cluster_key].isin(til_cell_types)
A = ad.obsp['spatial_connectivities']

analyzer = xu.SpatialTissueAnalyzer(radius=radius, n_perms=100) 

# STRATEGY B: Tissue-Adjusted Interaction (GAM Spatial)
# Use this to prove the effect isn't just because both are in the "stroma".
# res_df_b, summary_b = analyzer.run_meta(df_ecm, sample_col='sample', method='gam_spatial')

# STRATEGY C: Local Density Threshold (GAM Contrast: High vs Low)
# Use this to test if clustered TILs (>1) do more than isolated TILs (1).
# res_df_c, summary_c = analyzer.run_meta(df_ecm, sample_col='sample', method='gam_contrast')

res_df = {}
summary_df = {}
for target_col in ecm_cats:
    print(target_col)

    res_df[target_col], summary_df[target_col] = analyzer.run_meta(
        df_analysis, 
        sample_col='sample',
        source_col='is_TIL', 
        target_col=target_col, 
        method='spearman_spatial_perm',
        smooth_target=False,
    )

res_df = (
    pd.concat(res_df)
    .reset_index(level=1, drop=True)
    .reset_index().rename(columns={'index':'category'})
)

res_df = res_df.join(ad_rna.obs.set_index('sample')['donor'].drop_duplicates(),on='sample')

In [ ]:
save_path=Path(figures_dir/f'xenium/{panel}/{segmentation}/ecm_spearman.png')
save_path.parent.mkdir(exist_ok=True,parents=True)

dotplot(res_df,
    x_col='category',
    y_col='donor',
    diff_col='rho',
    x_order=res_df['category'].unique(),
    y_order=np.unique(res_df['donor']),
    figsize=(12, 5),
    # yticklabels=False,
    bbox_to_anchor_dot = (1.15, .8),
    min_dot_size=100,
    max_dot_size=400,
    save_path=save_path
)

# Plot

In [ ]:
s = 15
plot_cols = ['TIL_sum'] + ecm_cats
plot_ad = ad_rna_til_pos
df = plot_ad.obs[[sample_key]+plot_cols].copy()
df['x'] = plot_ad.obsm['spatial'][:, 0]
df['y'] = plot_ad.obsm['spatial'][:, 1]

for sample, sub_df in df.groupby('sample'):
    
    # 3. Setup Plot
    fig, axes = plt.subplots(2, 4, figsize=(30, 12))
    fig.suptitle(f"Sample: {sample}", fontsize=10)
    axes = axes.flat

    for i, plot_col in enumerate(plot_cols):
        if plot_col in ['TIL_sum']+ecm_cats:
            plot_df = sub_df.sort_values(plot_col)
            palette = 'viridis'
        elif plot_col in ['TIL_status', 'TIL_status_region']:
            plot_df = sub_df.sort_values(plot_col,ascending=False)
            palette = None
        elif plot_col == 'TIL_sum_cat':
            plot_df = sub_df.sort_values(plot_col)
            palette = None

        plot_df[plot_col] = plot_df[plot_col].clip(upper=plot_df[plot_col].quantile(0.98))
        p = sns.scatterplot(
            data=plot_df, 
            x='x', y='y', hue=plot_col, palette=palette,
            ax=axes[i], s=s, linewidth=0, alpha=0.8
        )

        axes[i].set_title(plot_col)
        axes[i].set_aspect('equal')
        axes[i].axis('off')
        if pd.api.types.is_numeric_dtype(plot_df[plot_col]):
            norm = plt.Normalize(plot_df[plot_col].min(), plot_df[plot_col].max())
            sm = plt.cm.ScalarMappable(cmap=palette, norm=norm)
            sm.set_array([])
            axes[i].get_legend().remove()
            axes[i].figure.colorbar(sm, ax=axes[i])
        else:
            axes[i].legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

    plt.tight_layout()

In [ ]:
s = 5
plot_cols = ['TIL_sum', 'TIL_status', 'TIL_status_region','TIL_sum_cat']
df = ad_rna.obs[[sample_key]+plot_cols].copy()
df['x'] = ad_rna.obsm['spatial'][:, 0]
df['y'] = ad_rna.obsm['spatial'][:, 1]

for sample, sub_df in df.groupby('sample'):
    
    # 3. Setup Plot
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    fig.suptitle(f"Sample: {sample}", fontsize=10)
    axes = axes.flat

    for i, plot_col in enumerate(plot_cols):
        if plot_col == 'TIL_sum':
            plot_df = sub_df.sort_values(plot_col)
            palette = 'viridis'
        elif plot_col in ['TIL_status', 'TIL_status_region']:
            plot_df = sub_df.sort_values(plot_col,ascending=False)
            palette = None
        elif plot_col == 'TIL_sum_cat':
            plot_df = sub_df.sort_values(plot_col)
            palette = None

        p = sns.scatterplot(
            data=plot_df, 
            x='x', y='y', hue=plot_col, palette=palette,
            ax=axes[i], s=s, linewidth=0, alpha=0.8
        )

        axes[i].set_title(plot_col)
        axes[i].set_aspect('equal')
        axes[i].axis('off')
        if pd.api.types.is_numeric_dtype(plot_df[plot_col]):
            norm = plt.Normalize(plot_df[plot_col].min(), plot_df[plot_col].max())
            sm = plt.cm.ScalarMappable(cmap=palette, norm=norm)
            sm.set_array([])
            axes[i].get_legend().remove()
            axes[i].figure.colorbar(sm, ax=axes[i])
        else:
            axes[i].legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_vars = ["CD3E", "CD8A", "CD4"]

for k, ad_ in xds[corr].items():
    print(k)
    ad_.obsm['spatial'] = xds['raw'][k][ad_.obs_names].obsm['spatial']
    sc.pl.embedding(ad_,'X_rna_umap_sample',color=ct_keys)
    sc.pl.embedding(ad_,'X_rna_umap_sample',color=plot_vars,vmax="p99")
    sc.pl.spatial(ad_,color=ct_keys,spot_size=20)
    sc.pl.spatial(ad_,color=plot_vars,spot_size=20,vmax="p99")

In [ ]:
warnings.filterwarnings("ignore", category=anndata.ImplicitModificationWarning)

outdir = "figures/spatial_by_sample"
Path(outdir).mkdir(exist_ok=True)

u_samples = ad_rna.obs["sample"].unique()
plot_vars = ["CD3E", "CD8A", "CD4", "CD45"]
doublet_keys = [f'celltype_level{i}_curated_spot_class' for i in range(1,5)]

for s in u_samples:
    print(s)

    ad_rna_sample = ad_rna[ad_rna.obs["sample"] == s].copy()
    ad_prot_sample = ad_prot[ad_prot.obs["sample"] == s].copy()

    rna_vars = [v for v in plot_vars if v in ad_rna_sample.var_names]
    prot_vars = [v for v in plot_vars if v in ad_prot_sample.var_names]

    sc.pl.spatial(ad_rna_sample,color=level_keys,spot_size=20,vmax="p99",show=False,save=f"_{s}_cell_types.png",ncols=1)
    # sc.pl.spatial(ad_rna_sample,color=doublet_keys,spot_size=20,vmax="p99",show=False,save=f"_{s}_doublet_status.png")

    # if rna_vars:
    #     sc.pl.spatial(ad_rna_sample,color=rna_vars,spot_size=20,vmax="p99",show=False,save=f"_{s}_RNA_markers.png")

    # if prot_vars:
    #     sc.pl.spatial(ad_prot_sample,color=prot_vars,spot_size=20,vmax="p99",show=False,save=f"_{s}_PROT_markers.png")


In [ ]:
k = list(xenium_paths['raw'])[0]

for k, p in xenium_paths['raw'].items():
    rna_umap = pd.read_csv(p/'analysis/umap/gene_expression_2_components/projection.csv',index_col=0)
    prot_umap = pd.read_csv(p/'analysis/umap/protein_expression_2_components/projection.csv',index_col=0)

    ad_ = xds['raw'][k][rna_umap.index].copy()
    ad_.obsm['X_rna_umap'] = rna_umap.values
    ad_.obsm['X_prot_umap'] = prot_umap.values
    xds['raw'][k] = ad_

In [ ]:
for k, ad_ in xds['raw'].items():
    print(k)
    sc.pl.embedding(ad_,'X_rna_umap',color=level_keys)
    sc.pl.embedding(ad_,'X_prot_umap',color=level_keys)